# Lab 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/geog-414/blob/master/book/labs/lab_07.ipynb)

## Submission requirements

1. Upload a screenshot of your map for each question.
2. Provide a link to your notebook on Colab. See instructions [here](https://geog-414.gishub.org/book/labs/instructions.html).

## Datasets

The datasets being used in the lab are listed below:

- [TIGER: US Census Counties](https://developers.google.com/earth-engine/datasets/catalog/TIGER_2018_Counties)
- [Landsat-9](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2)
- [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR)
- [NAIP](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ)

## Question 1

Create a fishnet with a 4-degree interval based on the extent of `[-112.5439, 34.0891, -85.0342, 49.6858]`. Use the fishnet to download the Landsat 7 image tiles by the fishnet using the `geemap.download_ee_image_tiles()` function. Relevant Earth Engine assets:

-   `ee.Image('LANDSAT/LE7_TOA_5YEAR/1999_2003')`

![](https://i.imgur.com/L1IH3fq.png)

In [2]:
import os
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'geedim'])

import ee
import geemap
from ipyleaflet import TileLayer, WidgetControl
from ipywidgets import HTML

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

bbox = [-112.5439, 34.0891, -85.0342, 49.6858]
region = ee.Geometry.BBox(*bbox)
fishnet = geemap.fishnet(region, h_interval=4.0, v_interval=4.0)

landsat = ee.Image('LANDSAT/LE7_TOA_5YEAR/1999_2003').select(
    ['B3', 'B2', 'B1']
)
vis_params = {'bands': ['B3', 'B2', 'B1'], 'min': 0.02, 'max': 0.35, 'gamma': 1.2}

m = geemap.Map(center=[41.8, -98.8], zoom=4)

esri_basemap = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)
kept_layers = [
    lyr
    for lyr in m.layers
    if 'openstreetmap' not in str(getattr(lyr, 'name', '')).lower()
    and 'osm' not in str(getattr(lyr, 'name', '')).lower()
    and 'openstreetmap' not in str(getattr(lyr, 'url', '')).lower()
]
if not any(
    'arcgisonline' in str(getattr(lyr, 'url', '')).lower() for lyr in kept_layers
):
  kept_layers.insert(0, esri_basemap)
m.layers = tuple(kept_layers)

m.addLayer(landsat, vis_params, 'Landsat 7 (1999-2003)')
m.addLayer(
    fishnet.style(color='yellow', fillColor='00000000', width=2),
    {},
    'Fishnet (4 deg)',
)

credit_widget = HTML(
    value=(
        '<div style="background: rgba(255,255,255,0.92); padding: 4px 10px;'
        ' border-radius: 4px; border: 1px solid #999; font-size: 12px;'
        ' font-weight: 600; color: #111;">Made by Mahdieh Zakizadeh</div>'
    )
)
m.add_control(WidgetControl(widget=credit_widget, position='bottomright'))

display(m)

out_dir = 'landsat7_tiles'
os.makedirs(out_dir, exist_ok=True)

geemap.download_ee_image_tiles(
    landsat,
    features=fishnet,
    out_dir=out_dir,
    prefix='tile_',
    crs='EPSG:4326',
    scale=250,
)

print('Downloaded tiles count:', len(os.listdir(out_dir)))

Map(center=[41.8, -98.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

1999_2003:   0%|          |0/6 tiles [00:00<?]

1999_2003:   0%|          |0/6 tiles [00:00<?]

1999_2003:   0%|          |0/6 tiles [00:00<?]

1999_2003:   0%|          |0/6 tiles [00:00<?]

1999_2003:   0%|          |0/6 tiles [00:00<?]

... (20 outputs left out)

1999_2003:   0%|          |0/6 tiles [00:00<?]

1999_2003:   0%|          |0/6 tiles [00:00<?]

1999_2003:   0%|          |0/6 tiles [00:00<?]

Downloaded 28 tiles in 92.10164499282837 seconds.
Downloaded tiles count: 28


## Question 2

Create annual cloud-free Landsat imagery for the years 2017-2023 for a US county of your choice. Download the images to your computer.

![](https://i.imgur.com/MN2UXHx.png)

In [7]:
import os
import shutil
import ee
import geemap
from google.colab import files
from ipyleaflet import TileLayer, WidgetControl
from ipywidgets import HTML

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

# Select a US county from TIGER/2018/Counties (Union County, TN)
counties = ee.FeatureCollection('TIGER/2018/Counties')
county = counties.filter(
    ee.Filter.And(
        ee.Filter.eq('NAME', 'Union'),
        ee.Filter.eq('STATEFP', '47'),
    )
)
county_geom = county.geometry()


# Cloud and cloud-shadow mask using QA_PIXEL for Landsat Collection 2 Level 2
def mask_l2_clouds(img):
  qa = img.select('QA_PIXEL')
  dilated_cloud = 1 << 1
  cirrus = 1 << 2
  cloud = 1 << 3
  cloud_shadow = 1 << 4
  snow = 1 << 5
  mask = (
      qa.bitwiseAnd(dilated_cloud)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus).eq(0))
      .And(qa.bitwiseAnd(cloud).eq(0))
      .And(qa.bitwiseAnd(cloud_shadow).eq(0))
      .And(qa.bitwiseAnd(snow).eq(0))
  )
  scaled = (
      img.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'])
      .multiply(0.0000275)
      .add(-0.2)
  )
  return scaled.updateMask(mask).copyProperties(img, ['system:time_start'])


# Build Landsat 8 & 9 collection (2017-2023)
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(county_geom)
l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(county_geom)
landsat_col = l8.merge(l9).map(mask_l2_clouds)

years = list(range(2017, 2024))
annual_composites = {}

for yr in years:
  start_date = ee.Date.fromYMD(yr, 1, 1)
  end_date = ee.Date.fromYMD(yr, 12, 31)
  annual_img = (
      landsat_col.filterDate(start_date, end_date)
      .median()
      .clip(county_geom)
      .set('year', yr)
  )
  annual_composites[yr] = annual_img

# Display map with 2023 False Color composite and county boundary
m = geemap.Map(center=[36.38, -83.84], zoom=10)

esri_basemap = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)
kept_layers = [
    lyr
    for lyr in m.layers
    if 'openstreetmap' not in str(getattr(lyr, 'name', '')).lower()
    and 'osm' not in str(getattr(lyr, 'name', '')).lower()
    and 'openstreetmap' not in str(getattr(lyr, 'url', '')).lower()
]
if not any(
    'arcgisonline' in str(getattr(lyr, 'url', '')).lower() for lyr in kept_layers
):
  kept_layers.insert(0, esri_basemap)
m.layers = tuple(kept_layers)

vis_false_color = {
    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],
    'min': 0.02,
    'max': 0.35,
    'gamma': 1.2,
}
m.addLayer(annual_composites[2023], vis_false_color, 'Landsat 2023 False Color')
m.addLayer(
    county.style(color='blue', fillColor='00000000', width=2),
    {},
    'Union County Boundary',
)

# Add attribution label on the map
credit_widget = HTML(
    value=(
        '<div style="background: rgba(255,255,255,0.92); padding: 4px 10px;'
        ' border-radius: 4px; border: 1px solid #999; font-size: 12px;'
        ' font-weight: 600; color: #111;">Made by Mahdieh Zakizadeh</div>'
    )
)
m.add_control(WidgetControl(widget=credit_widget, position='bottomright'))

display(m)

# Export annual cloud-free GeoTIFFs and zip for downloading to computer
out_dir = 'landsat_annual_2017_2023'
os.makedirs(out_dir, exist_ok=True)

for yr, img in annual_composites.items():
  out_tif = os.path.join(out_dir, f'landsat_union_county_{yr}.tif')
  geemap.ee_export_image(
      img.select(['SR_B5', 'SR_B4', 'SR_B3', 'SR_B2']),
      filename=out_tif,
      scale=120,
      region=county_geom,
      file_per_band=False,
  )

zip_base = 'landsat_union_county_2017_2023'
zip_path = shutil.make_archive(zip_base, 'zip', out_dir)
print('Exported files:', sorted(os.listdir(out_dir)))
files.download(zip_path)

Map(center=[36.38, -83.84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2017.tif
Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2018.tif
Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2019.tif
Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2020.tif
Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2021.tif
Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2022.tif
Generating URL ...
Please wait ...
Data downloaded to /content/landsat_annual_2017_2023/landsat_union_county_2023.tif
Exported files: ['landsat_union_county_2017.tif', 'landsat_union_county_2018.tif', 'landsat_union_county_2019.tif', 'landsat_union_county_2020.tif', 'landsat_union_county_202

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Question 3

Create annual cloud-free Sentinel-2 imagery for the years 2017-2023 for a US county of your choice. Download the images to your computer. You can download a coarse resolution image to speed up the download process. Narrow down the date range (e.g., summer months) to reduce the number of images, which can avoid memory errors.

![](https://i.imgur.com/r5RQlEJ.png)

In [9]:
import os
import shutil
import ee
import geemap
from google.colab import files
from ipyleaflet import TileLayer, WidgetControl
from ipywidgets import HTML

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

counties = ee.FeatureCollection('TIGER/2018/Counties')
county = counties.filter(
    ee.Filter.And(
        ee.Filter.eq('NAME', 'Union'),
        ee.Filter.eq('STATEFP', '47'),
    )
)
county_geom = county.geometry()


def mask_s2_clouds(img):
  qa = img.select('QA60')
  cloud_bit = 1 << 10
  cirrus_bit = 1 << 11
  mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
  scaled = img.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).multiply(0.0001)
  return scaled.updateMask(mask).copyProperties(img, ['system:time_start'])


s2_col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(county_geom)
    .filter(ee.Filter.calendarRange(6, 8, 'month'))
    .map(mask_s2_clouds)
)

years = list(range(2017, 2024))
annual_s2 = {}

for yr in years:
  start_date = ee.Date.fromYMD(yr, 6, 1)
  end_date = ee.Date.fromYMD(yr, 8, 31)
  yr_img = (
      s2_col.filterDate(start_date, end_date)
      .median()
      .clip(county_geom)
      .set('year', yr)
  )
  annual_s2[yr] = yr_img

m = geemap.Map(center=[36.38, -83.84], zoom=10)

esri_basemap = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)
kept_layers = [
    lyr
    for lyr in m.layers
    if 'openstreetmap' not in str(getattr(lyr, 'name', '')).lower()
    and 'osm' not in str(getattr(lyr, 'name', '')).lower()
    and 'openstreetmap' not in str(getattr(lyr, 'url', '')).lower()
]
if not any(
    'arcgisonline' in str(getattr(lyr, 'url', '')).lower() for lyr in kept_layers
):
  kept_layers.insert(0, esri_basemap)
m.layers = tuple(kept_layers)

vis_s2 = {
    'bands': ['B11', 'B8', 'B4'],
    'min': 0.03,
    'max': 0.45,
    'gamma': 1.15,
}
m.addLayer(
    annual_s2[2023], vis_s2, 'Sentinel-2 Summer 2023 (SWIR/NIR/Red)'
)
m.addLayer(
    county.style(color='blue', fillColor='00000000', width=2),
    {},
    'Union County Boundary',
)

credit_widget = HTML(
    value=(
        '<div style="background: rgba(255,255,255,0.92); padding: 4px 10px;'
        ' border-radius: 4px; border: 1px solid #999; font-size: 12px;'
        ' font-weight: 600; color: #111;">Made by Mahdieh Zakizadeh</div>'
    )
)
m.add_control(WidgetControl(widget=credit_widget, position='bottomright'))

display(m)

out_dir = 'sentinel2_summer_2017_2023'
os.makedirs(out_dir, exist_ok=True)

for yr, img in annual_s2.items():
  out_tif = os.path.join(out_dir, f'sentinel2_union_county_summer_{yr}.tif')
  geemap.ee_export_image(
      img.select(['B11', 'B8', 'B4', 'B3', 'B2']),
      filename=out_tif,
      scale=120,
      region=county_geom,
      file_per_band=False,
  )

zip_base = 'sentinel2_union_county_summer_2017_2023'
zip_path = shutil.make_archive(zip_base, 'zip', out_dir)
print('Exported files:', sorted(os.listdir(out_dir)))
files.download(zip_path)

Map(center=[36.38, -83.84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2017.tif
Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2018.tif
Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2019.tif
Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2020.tif
Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2021.tif
Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2022.tif
Generating URL ...
Please wait ...
Data downloaded to /content/sentinel2_summer_2017_2023/sentinel2_union_county_summer_2023.tif
Exported files: ['sentinel2_union_county_summer_2017.tif', 'sentinel2_union_county_summer_2018.ti

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Question 4

Create annual cloud-free NAIP imagery for the years 2010-2023 for a US county of your choice. Download the images to your computer. You can download a coarse resolution image to speed up the download process.

![](https://i.imgur.com/h66FC8h.png)

In [10]:
import os
import shutil
import ee
import geemap
from google.colab import files
from ipyleaflet import TileLayer, WidgetControl
from ipywidgets import HTML

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

counties = ee.FeatureCollection('TIGER/2018/Counties')
county = counties.filter(
    ee.Filter.And(
        ee.Filter.eq('NAME', 'Union'),
        ee.Filter.eq('STATEFP', '47'),
    )
)
county_geom = county.geometry()

naip_col = (
    ee.ImageCollection('USDA/NAIP/DOQQ')
    .filterBounds(county_geom)
    .filterDate('2010-01-01', '2023-12-31')
)

years = list(range(2010, 2024))
annual_naip = {}

for yr in years:
  yr_col = naip_col.filter(ee.Filter.calendarRange(yr, yr, 'year'))
  count = yr_col.size().getInfo()
  if count > 0:
    yr_img = (
        yr_col.select(['R', 'G', 'B', 'N'])
        .median()
        .clip(county_geom)
        .set('year', yr)
    )
    annual_naip[yr] = yr_img

m = geemap.Map(center=[36.38, -83.84], zoom=10)

esri_basemap = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)
kept_layers = [
    lyr
    for lyr in m.layers
    if 'openstreetmap' not in str(getattr(lyr, 'name', '')).lower()
    and 'osm' not in str(getattr(lyr, 'name', '')).lower()
    and 'openstreetmap' not in str(getattr(lyr, 'url', '')).lower()
]
if not any(
    'arcgisonline' in str(getattr(lyr, 'url', '')).lower() for lyr in kept_layers
):
  kept_layers.insert(0, esri_basemap)
m.layers = tuple(kept_layers)

latest_yr = max(annual_naip.keys())
vis_naip = {'bands': ['N', 'R', 'G'], 'min': 0, 'max': 255}
m.addLayer(
    annual_naip[latest_yr],
    vis_naip,
    f'NAIP {latest_yr} False Color (NIR/R/G)',
)
m.addLayer(
    county.style(color='blue', fillColor='00000000', width=2),
    {},
    'Union County Boundary',
)

credit_widget = HTML(
    value=(
        '<div style="background: rgba(255,255,255,0.92); padding: 4px 10px;'
        ' border-radius: 4px; border: 1px solid #999; font-size: 12px;'
        ' font-weight: 600; color: #111;">Made by Mahdieh Zakizadeh</div>'
    )
)
m.add_control(WidgetControl(widget=credit_widget, position='bottomright'))

display(m)

out_dir = 'naip_annual_2010_2023'
os.makedirs(out_dir, exist_ok=True)

for yr, img in annual_naip.items():
  out_tif = os.path.join(out_dir, f'naip_union_county_{yr}.tif')
  geemap.ee_export_image(
      img.select(['R', 'G', 'B', 'N']),
      filename=out_tif,
      scale=30,
      region=county_geom,
      file_per_band=False,
  )

zip_base = 'naip_union_county_2010_2023'
zip_path = shutil.make_archive(zip_base, 'zip', out_dir)
print('Available NAIP years:', sorted(annual_naip.keys()))
print('Exported files:', sorted(os.listdir(out_dir)))
files.download(zip_path)# Add your code here.

Map(center=[36.38, -83.84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2010.tif
Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2012.tif
Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2014.tif
Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2016.tif
Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2018.tif
Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2021.tif
Generating URL ...
Please wait ...
Data downloaded to /content/naip_annual_2010_2023/naip_union_county_2023.tif
Available NAIP years: [2010, 2012, 2014, 2016, 2018, 2021, 2023]
Exported files: ['naip_union_county_2010.tif', 'naip_union_county_2012.tif', 'naip_union_county_2014.tif', 'naip_union_county_2016.tif', 'naip_union_co

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Question 5

Download a US county of your choice and save it as a shapefile or GeoJSON file.

![](https://i.imgur.com/PuK2Vp3.png)

In [1]:
import os
import shutil
import ee
import geemap
from google.colab import files
from ipyleaflet import TileLayer, WidgetControl
from ipywidgets import HTML

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

counties = ee.FeatureCollection('TIGER/2018/Counties')
county = counties.filter(
    ee.Filter.And(
        ee.Filter.eq('NAME', 'Union'),
        ee.Filter.eq('STATEFP', '47'),
    )
)

m = geemap.Map(center=[36.38, -83.84], zoom=10)

esri_basemap = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)
kept_layers = [
    lyr
    for lyr in m.layers
    if 'openstreetmap' not in str(getattr(lyr, 'name', '')).lower()
    and 'osm' not in str(getattr(lyr, 'name', '')).lower()
    and 'openstreetmap' not in str(getattr(lyr, 'url', '')).lower()
]
if not any(
    'arcgisonline' in str(getattr(lyr, 'url', '')).lower() for lyr in kept_layers
):
  kept_layers.insert(0, esri_basemap)
m.layers = tuple(kept_layers)

m.addLayer(
    county.style(color='blue', fillColor='3388ff33', width=2),
    {},
    'Union County, TN',
)

credit_widget = HTML(
    value=(
        '<div style="background: rgba(255,255,255,0.92); padding: 4px 10px;'
        ' border-radius: 4px; border: 1px solid #999; font-size: 12px;'
        ' font-weight: 600; color: #111;">Made by Mahdieh Zakizadeh</div>'
    )
)
m.add_control(WidgetControl(widget=credit_widget, position='bottomright'))

display(m)

# Save as GeoJSON
geojson_path = 'union_county_tn.geojson'
geemap.ee_to_geojson(county, filename=geojson_path)

# Save as Shapefile inside a ZIP archive
shp_dir = 'union_county_tn_shp'
os.makedirs(shp_dir, exist_ok=True)
shp_path = os.path.join(shp_dir, 'union_county_tn.shp')
geemap.ee_to_shp(county, filename=shp_path)
shp_zip_path = shutil.make_archive('union_county_tn_shapefile', 'zip', shp_dir)

print('Saved files:', geojson_path, shp_zip_path)
files.download(geojson_path)
files.download(shp_zip_path)# Add your code here.

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=XPEqtz2rAbXGd42spgz5cojrXoMWeXMoJsYqM5Khh4U&tc=lb0B93lEsDZz8LTWQD3KjldU5wC-WwHGYzCXQM9s6d8&cc=Ed-2D-Qn_J0qIDHYjCNvGW8swufho2Kd5yHWdWOiDfI

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AUn6Hd-x04DkShNU1d20vnZV_agcsx0LTIisdkL1zNnLb2D8wEo3UZqihyI

Successfully saved authorization token.


Map(center=[36.38, -83.84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

Saved files: union_county_tn.geojson /content/union_county_tn_shapefile.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>